# US bysykkel data

- data is from 2024
- Contact person is Christoffer Bakken Åkre (US)

## Imports

In [165]:
import pandas as pd
from pathlib import Path
import os
import numpy as np
from geopy.distance import geodesic
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
import statsmodels.api as sm
import statsmodels.formula.api as smf

## Load data from raw_files

In [166]:
# Directories setup
notebook_dir = Path(os.getcwd())
raw_trip_files_dir = notebook_dir / 'trip_data' / 'raw_files'
raw_maintenance_files_dir = notebook_dir / 'maintenance_data' / 'raw_files'

_trip_files = sorted(raw_trip_files_dir.glob('*.parquet'))
_maintenance_files = sorted(raw_maintenance_files_dir.glob('*.csv'))

print(f"Found {len(_trip_files)} parquet files in {raw_trip_files_dir}")
print(f"Found {len(_maintenance_files)} CSV files in {raw_maintenance_files_dir}")

# --- Load Trip Data (Keeping the logs) ---
trip_dfs = []
for file in _trip_files:
    print(f"Loading {file.name}...")
    df = pd.read_parquet(file)
    trip_dfs.append(df)

# Combine into one master trip dataframe
trip_data = pd.concat(trip_dfs, ignore_index=True) if trip_dfs else pd.DataFrame()

# --- Load Maintenance Data (Separating into Damage, Maint, Repair) ---
damage_list = []
maint_list = []
repair_list = []

for file in _maintenance_files:
    print(f"Loading {file.name}...")
    df = pd.read_csv(file)
    
    fname = file.name.lower()
    if 'damage' in fname:
        damage_list.append(df)
    elif 'repair' in fname:
        repair_list.append(df)
    else:
        # Assuming files not marked damage/repair are general maintenance
        maint_list.append(df)

# Create the separate DataFrames
damage_data = pd.concat(damage_list, ignore_index=True) if damage_list else pd.DataFrame()
maintenance_data = pd.concat(maint_list, ignore_index=True) if maint_list else pd.DataFrame()
repair_data = pd.concat(repair_list, ignore_index=True) if repair_list else pd.DataFrame()

print("-" * 30)
print(f"Trips loaded: {len(trip_data)} rows")
print(f"Damage logs:  {len(damage_data)} rows")
print(f"Maint logs:   {len(maintenance_data)} rows")
print(f"Repair logs:  {len(repair_data)} rows")



Found 9 parquet files in c:\Users\Minamsj\FOMOsim\policies\sjovik_sund\US_data\trip_data\raw_files
Found 3 CSV files in c:\Users\Minamsj\FOMOsim\policies\sjovik_sund\US_data\maintenance_data\raw_files
Loading 000000000000.parquet...
Loading 000000000001.parquet...
Loading 000000000002.parquet...
Loading 000000000003.parquet...
Loading 000000000004.parquet...
Loading 000000000005.parquet...
Loading 000000000006.parquet...
Loading 000000000007.parquet...
Loading 000000000008.parquet...
Loading asset_damage.csv...
Loading asset_maintenance.csv...
Loading asset_repair.csv...
------------------------------
Trips loaded: 5748944 rows
Damage logs:  756565 rows
Maint logs:   363324 rows
Repair logs:  250815 rows


# Null values in data

In [167]:
"""def print_null_report(df_name, df):
    '''Generates a clean table of null values for a dataframe.'''
    print(f"\n{'='*20} {df_name.upper()} ({len(df)} rows) {'='*20}")
    
    # Calculate counts and percentages
    null_counts = df.isnull().sum()
    total_rows = len(df)
    
    table_data = []
    for col, count in null_counts.items():
        # Calculate percentage
        percent = (count / total_rows) * 100
        
        # Add a status flag
        if count == 0:
            status = " " # Clean
        elif percent > 90:
            status = "CRITICAL" # Mostly empty
        else:
            status = "Check"
            
        table_data.append([col, count, f"{percent:.1f}%", status])

    # Sort: Show columns with the most missing values first
    table_data.sort(key=lambda x: x[1], reverse=True)

    # Print using tabulate
    print(tabulate(table_data, 
                   headers=["Column", "Missing Count", "% Missing", "Status"], 
                   tablefmt="psql"))

# --- Execution ---
datasets = {
    "Damage Data": damage_data,
    "Maintenance Data": maintenance_data,
    "Repair Data": repair_data,
    "Trip Data": trip_data
}

for name, df in datasets.items():
    print_null_report(name, df)"""

'def print_null_report(df_name, df):\n    \'\'\'Generates a clean table of null values for a dataframe.\'\'\'\n    print(f"\n{\'=\'*20} {df_name.upper()} ({len(df)} rows) {\'=\'*20}")\n\n    # Calculate counts and percentages\n    null_counts = df.isnull().sum()\n    total_rows = len(df)\n\n    table_data = []\n    for col, count in null_counts.items():\n        # Calculate percentage\n        percent = (count / total_rows) * 100\n\n        # Add a status flag\n        if count == 0:\n            status = " " # Clean\n        elif percent > 90:\n            status = "CRITICAL" # Mostly empty\n        else:\n            status = "Check"\n\n        table_data.append([col, count, f"{percent:.1f}%", status])\n\n    # Sort: Show columns with the most missing values first\n    table_data.sort(key=lambda x: x[1], reverse=True)\n\n    # Print using tabulate\n    print(tabulate(table_data, \n                   headers=["Column", "Missing Count", "% Missing", "Status"], \n                   tabl

# Trip data preprocessing

### Drop columns

In [168]:
#remove columns with certain names
columns_to_remove = ['position_wkt','trip_unlock_method','user_id', 'product_id', 'product_name', 'product_price', 'product_sales_channel', 'product_sales_locale', 'sale_from_value_code']  # specify columns to remove

trip_data = trip_data.drop(columns=[col for col in columns_to_remove if col in trip_data.columns])

### Integer conversion

In [169]:
# make columns integer data type
cols_to_int = ['trip_end_dock_group_id', 'trip_end_dock_number']
for col in cols_to_int:
    if col in trip_data.columns:
        trip_data[col] = pd.to_numeric(trip_data[col], errors='coerce').fillna(0).astype(int)

### Datetime conversion

In [170]:
# List of dataframes and their respective time columns to convert
datasets = {
    'Trip': (trip_data, ['position_at', 'trip_started_at', 'trip_ended_at'])
}

for name, (df, cols) in datasets.items():
    if df is not None:
        for col in cols:
            if col in df.columns:
                # Convert to datetime, strip timezone if present, and force nanosecond [ns]
                df[col] = pd.to_datetime(df[col], errors='coerce').dt.tz_localize(None).astype('datetime64[ns]')
        print(f"{name} Data time columns converted to datetime64[ns].")

# Verify the types
print(f"Trip data types:\n{trip_data.dtypes}\n")

Trip Data time columns converted to datetime64[ns].
Trip data types:
vehicle_id                              int64
position_at                    datetime64[ns]
position_latitude                     float64
position_longitude                    float64
position_accuracy                       int64
trip_started_at                datetime64[ns]
trip_ended_at                  datetime64[ns]
trip_id                                 int64
trip_state                             object
trip_start_dock_number                  int64
trip_start_dock_group_id                int64
trip_start_dock_group_title            object
trip_end_dock_number                    int64
trip_end_dock_group_id                  int64
trip_end_dock_group_title              object
dtype: object



## Unique values of stations

In [171]:
import pandas as pd
from tabulate import tabulate

# 1. Define the relevant columns
station_id_col = 'trip_start_dock_group_id'
station_name_col = 'trip_start_dock_group_title'

# 2. Extract Unique Stations
# We select the two columns and drop duplicate rows to get unique pairs
unique_stations = (
    trip_data[[station_id_col, station_name_col]]
    .drop_duplicates()
    .rename(columns={
        station_id_col: 'Station ID', 
        station_name_col: 'Station Name'
    })
)

# 3. Sort by Station ID for better readability
unique_stations = unique_stations.sort_values(by='Station ID')

# 4. Print the result
print(f"\n--- UNIQUE STATIONS FOUND: {len(unique_stations)} ---")
print(tabulate(unique_stations, headers='keys', tablefmt='psql', showindex=False))


--- UNIQUE STATIONS FOUND: 70 ---
+--------------+---------------------------------+
|   Station ID | Station Name                    |
|--------------+---------------------------------|
|            1 | Dokkparken                      |
|            4 | Dalen hageby                    |
|            6 | Lademoparken                    |
|            9 | Brattørkaia                     |
|           10 | Sorgenfri                       |
|           14 | Rådhuset                        |
|           16 | Rosenborg skole                 |
|           18 | Møllenberg                      |
|           19 | Jomfrugateallmenningen          |
|           20 | Nordre gate                     |
|           21 | Bakke bru                       |
|           25 | Thornesparken                   |
|           28 | Kongens gate                    |
|           29 | Singsaker                       |
|           30 | St. Olavs gate                  |
|           31 | Tollboden                     

## Trip distance column

In [172]:
def add_trip_distances(df):
    """
    Adds 'segment_distance_m' (distance from previous point) 
    and 'total_trip_distance_meters' (total length of the trip)
    to the original dataframe.
    """
    
    # 1. Sort Data (Crucial for correct sequential calculation)
    # We sort in place or reassign to ensure the order is correct for shifting
    df = df.sort_values(by=['trip_id', 'position_at']).copy()

    # 2. Vectorized Haversine Calculation setup
    # Shift to get the "previous" point's coordinates per trip
    df['prev_lat'] = df.groupby('trip_id')['position_latitude'].shift(1)
    df['prev_lon'] = df.groupby('trip_id')['position_longitude'].shift(1)

    # 3. Calculate Distance for every row (Segment Distance)
    R = 6371000  # Earth radius in meters
    
    # We use numpy where to handle the first row of each trip (where prev is NaN)
    # This avoids dropping rows.
    lat1 = np.radians(df['prev_lat'])
    lon1 = np.radians(df['prev_lon'])
    lat2 = np.radians(df['position_latitude'])
    lon2 = np.radians(df['position_longitude'])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    # Calculate distance, filling NaNs (start of trips) with 0
    df['segment_distance_m'] = (R * c).fillna(0)

    # 4. Calculate Total Trip Distance (The new column you want)
    df['total_trip_distance_meters'] = df.groupby('trip_id')['segment_distance_m'].transform('sum')
    
    # Optional: Clean up helper columns if you don't want them cluttering the df
    df.drop(columns=['prev_lat', 'prev_lon'], inplace=True)

    return df

trip_data = add_trip_distances(trip_data)


In [173]:
# Group trips by trip_id and get the last row of each trip to have one row per trip with total distance
trip_data = trip_data.groupby('trip_id').last().reset_index()
trip_data = trip_data.drop(columns=['position_latitude', 'position_longitude', 'position_at'])

# Maintenance data preprocessing

## Remove columns

In [174]:
# remove columns with certain names in damage, maintenance and repair data
columns_to_remove = ['asset_model_name']  # specify columns to remove
damage_data = damage_data.drop(columns=[col for col in columns_to_remove if col in damage_data.columns])
maintenance_data = maintenance_data.drop(columns=[col for col in columns_to_remove if col in maintenance_data.columns])
repair_data = repair_data.drop(columns=[col for col in columns_to_remove if col in repair_data.columns])    

## Convert fields

In [175]:
# Print the data types of all the fields in the data
print(f"Damage data types:\n{damage_data.dtypes}\n")
print(f"Maintenance data types:\n{maintenance_data.dtypes}\n")
print(f"Repair data types:\n{repair_data.dtypes}\n")
print(f"Trip data types:\n{trip_data.dtypes}\n")

Damage data types:
vehicle_id                        int64
asset_model_id                    int64
vehicle_category                 object
case                             object
damage_id                         int64
created_at                       object
comment                          object
damage_type_id                    int64
damage_type_name                 object
set_vehicle_unavailable            bool
reported_by_administrator_id    float64
reported_by_user_id             float64
repeats                           int64
resolved_at                      object
asset_maintenance_id              int64
dtype: object

Maintenance data types:
vehicle_id                         int64
asset_model_id                     int64
vehicle_category                  object
case                              object
asset_maintenance_id               int64
started_at                        object
completed_at                      object
comment                           object
completed_by_a

### To datetime
- Changing all date fields to be on date time


In [176]:
# List of dataframes and their respective time columns to convert
datasets = {
    'Damage': (damage_data, ['created_at', 'resolved_at']),
    'Maintenance': (maintenance_data, ['started_at', 'completed_at']),
    'Repair': (repair_data, ['created_at', 'started_at', 'completed_at']),
}

for name, (df, cols) in datasets.items():
    if df is not None:
        for col in cols:
            if col in df.columns:
                # Convert to datetime, strip timezone if present, and force nanosecond [ns]
                df[col] = pd.to_datetime(df[col], errors='coerce').dt.tz_localize(None).astype('datetime64[ns]')
        print(f"{name} Data time columns converted to datetime64[ns].")

# Verify the types
print("-" * 30)
print(f"Damage data types:\n{damage_data.dtypes}\n")
print(f"Maintenance data types:\n{maintenance_data.dtypes}\n")
print(f"Repair data types:\n{repair_data.dtypes}\n")

Damage Data time columns converted to datetime64[ns].
Maintenance Data time columns converted to datetime64[ns].
Repair Data time columns converted to datetime64[ns].
------------------------------
Damage data types:
vehicle_id                               int64
asset_model_id                           int64
vehicle_category                        object
case                                    object
damage_id                                int64
created_at                      datetime64[ns]
comment                                 object
damage_type_id                           int64
damage_type_name                        object
set_vehicle_unavailable                   bool
reported_by_administrator_id           float64
reported_by_user_id                    float64
repeats                                  int64
resolved_at                     datetime64[ns]
asset_maintenance_id                     int64
dtype: object

Maintenance data types:
vehicle_id                             

### To integer
Turning the following fields integer:
- vehicle_id 
- asset_model_id 
- damage_id
- damage_type_id
- reported_by_administrator_id                
- reported_by_user_id                        
- repeats                                      
- asset_maintenance_id                        
- completed_by_administrator_id              
- repair_id                                  
- repair_type_id                             
- replaced_part_id                            
- repaired_by_administrator_id                

In [177]:
# turn the following fields into int64 in all maint/repair/damage data
cols_to_int = [
    'vehicle_id', 
    'asset_model_id', 
    'damage_id', 
    'damage_type_id', 
    'reported_by_administrator_id',
    'reported_by_user_id',
    'repeats', 
    'asset_maintenance_id', 
    'completed_by_administrator_id',
    'repair_id', 
    'repair_type_id', 
    'replaced_part_id',
    'repaired_by_administrator_id'
]

for col in cols_to_int:
    if col in damage_data.columns:
        damage_data[col] = pd.to_numeric(damage_data[col], errors='coerce').fillna(0).astype(int)
    if col in maintenance_data.columns:
        maintenance_data[col] = pd.to_numeric(maintenance_data[col], errors='coerce').fillna(0).astype(int)
    if col in repair_data.columns:
        repair_data[col] = pd.to_numeric(repair_data[col], errors='coerce').fillna(0).astype(int)

# print data types again to confirm
print(f"Damage data types after conversion:\n{damage_data.dtypes}\n")
print(f"Maintenance data types after conversion:\n{maintenance_data.dtypes}\n")
print(f"Repair data types after conversion:\n{repair_data.dtypes}\n")
print(f"Trip data types after conversion:\n{trip_data.dtypes}\n")

Damage data types after conversion:
vehicle_id                               int64
asset_model_id                           int64
vehicle_category                        object
case                                    object
damage_id                                int64
created_at                      datetime64[ns]
comment                                 object
damage_type_id                           int64
damage_type_name                        object
set_vehicle_unavailable                   bool
reported_by_administrator_id             int64
reported_by_user_id                      int64
repeats                                  int64
resolved_at                     datetime64[ns]
asset_maintenance_id                     int64
dtype: object

Maintenance data types after conversion:
vehicle_id                                int64
asset_model_id                            int64
vehicle_category                         object
case                                     object
asset_maint

## Filter values

### On case = "Trondheim"

In [178]:
# Filter damage, repair and maintenance data to only include records where case = "trondheim"
damage_data = damage_data[damage_data['case'].str.lower() == 'trondheim']
maintenance_data = maintenance_data[maintenance_data['case'].str.lower() == 'trondheim']      
repair_data = repair_data[repair_data['case'].str.lower() == 'trondheim']   

print(f"After filtering for Trondheim:")
print(f"Damage logs:  {len(damage_data)} rows")
print(f"Maint logs:   {len(maintenance_data)} rows")
print(f"Repair logs:  {len(repair_data)} rows")

After filtering for Trondheim:
Damage logs:  139266 rows
Maint logs:   73932 rows
Repair logs:  51867 rows


### Add list of related damages to each maintenance record

In [179]:

# --- 1. FILTER & PREPARE ---
# Filter out damages that aren't linked to maintenance
linked_damages = damage_data[damage_data['asset_maintenance_id'].notna()].copy()

# Ensure damage_id is an integer
linked_damages['damage_id'] = linked_damages['damage_id'].fillna(0).astype(int)

# --- 2. GROUP & LIST ---
# Group by maintenance ID and aggregate damage IDs into a list
damage_lists = linked_damages.groupby('asset_maintenance_id')['damage_id'].apply(list).reset_index()
damage_lists.rename(columns={'damage_id': 'related_damages'}, inplace=True)

# --- 3. MERGE ---
# We use 'left' join to keep maintenance records even if they have no linked damages
maintenance_data = maintenance_data.merge(damage_lists, on='asset_maintenance_id', how='left')

# Fill NaN lists with empty lists []
maintenance_data['related_damages'] = maintenance_data['related_damages'].apply(
    lambda d: d if isinstance(d, list) else []
)

# --- 4. PRINT THE BOTTOM 400 ROWS ---
print("\n--- LAST 400 MAINTENANCE RECORDS WITH LINKED DAMAGES ---")

print(tabulate(maintenance_data[['asset_maintenance_id', 'related_damages']].tail(1000), 
               headers=['Maintenance ID', 'Linked Damage IDs'], 
               tablefmt='psql', 
               showindex=False))


--- LAST 400 MAINTENANCE RECORDS WITH LINKED DAMAGES ---
+------------------+--------------------------------------------------------------------------------------------------------------+
|   Maintenance ID | Linked Damage IDs                                                                                            |
|------------------+--------------------------------------------------------------------------------------------------------------|
|           780641 | [1799614]                                                                                                    |
|           780644 | [1799618]                                                                                                    |
|           780652 | [1799628]                                                                                                    |
|           780656 | [1799634]                                                                                                    |
|           780658

### On 2024

In [180]:
# Filter all damage, maintnance and repair on the case where created_at/started_at is in 2024
damage_data = damage_data[pd.to_datetime(damage_data['created_at']).dt.year == 2024]
maintenance_data = maintenance_data[pd.to_datetime(maintenance_data['started_at']).dt.year == 2024]
repair_data = repair_data[pd.to_datetime(repair_data['created_at']).dt.year == 2024]
print(f"After filtering for 2024:")
print(f"Damage logs:  {len(damage_data)} rows")
print(f"Maint logs:   {len(maintenance_data)} rows")
print(f"Repair logs:  {len(repair_data)} rows")



After filtering for 2024:
Damage logs:  37689 rows
Maint logs:   22786 rows
Repair logs:  9839 rows


### "unresponsive controller" outside operating hours

In [181]:
# 1. Setup: Extract the hour
damage_data['created_hour'] = damage_data['created_at'].dt.hour

# 2. Statistics BEFORE filtering
initial_count = len(damage_data)
print(f"--- BEFORE FILTERING ---")
print(f"Total Damage Logs: {initial_count:,}")

# Optional: Preview specific rows targeted for removal
target_rows = damage_data[
    (damage_data['damage_type_name'] == "Unresponsive Controller") & 
    (damage_data['created_hour'] < 6)
]
print(f"Identified {len(target_rows):,} 'Unresponsive Controller' logs between 00:00 and 06:00.")
if len(target_rows) > 0:
    print("Sample of rows to be removed:")
    print(target_rows[['created_at', 'damage_type_name']].head(5).to_string(index=False))

# 3. APPLY FILTER
# Logic: Keep rows where NOT (Type is "Unresponsive Controller" AND Hour is < 6)
damage_data = damage_data[
    ~((damage_data['damage_type_name'] == "Unresponsive Controller") & 
      (damage_data['created_hour'] < 6))
]

# 4. Statistics AFTER filtering
final_count = len(damage_data)
removed_count = initial_count - final_count

print(f"\n--- FILTERING RESULTS ---")
print(f"Rows Removed:       {removed_count:,}")
print(f"Remaining Logs:     {final_count:,}")
print("=" * 30)

--- BEFORE FILTERING ---
Total Damage Logs: 37,689
Identified 24,844 'Unresponsive Controller' logs between 00:00 and 06:00.
Sample of rows to be removed:
             created_at        damage_type_name
2024-03-16 04:30:03.620 Unresponsive Controller
2024-03-31 05:00:01.577 Unresponsive Controller
2024-03-31 05:00:01.674 Unresponsive Controller
2024-03-31 05:00:01.675 Unresponsive Controller
2024-03-31 05:00:02.095 Unresponsive Controller

--- FILTERING RESULTS ---
Rows Removed:       24,844
Remaining Logs:     12,845


## Split damages into mechanical and system errors

In [182]:
# p# --- GROUP 1: MECHANICAL (Dependent on Usage / Distance) ---
# Model these using: Poisson(Distance)
mechanical_damages = [
    'Bremse(r)',                   # Brakes (Physical wear)
    'Brake(s) need adjustment',    # Brakes (Cable stretch/wear)
    'Hjul',                        # Wheels (Impact/wear)
    'Lite luft',                   # Low Air (Punctures/Valve leaks)
    'Gir',                         # Gears (Mechanical wear)
    'Belte / Belt',                # Drive Belt (Friction wear)
    'Pedaler',                     # Pedals (Bearings/Impact)
    'Cranck bearing/bottom bracket ', # Cranks (Bearing fatigue)
    'Styre',                       # Handlebar (Impact/Fatigue)
    'Styrelager',                  # Headset Bearings (Vibration)
    'Sete',                        # Seat (Physical damage)
    'Setepinneklemme',             # Seat Clamp (Mechanical stress)
    'Støtte',                      # Kickstand (Spring/Hinge wear)
    'Skjerm(er)',                  # Fenders (Vibration/Impact)
    'Ringeklokke',                 # Bell (Mechanical spring)
    'Basket',                      # Basket (Load/Impact)
    'Lys',                         # Lights (Vibration/Wiring fatigue)
    'Frame'                        # Frame (Structural stress)
]

# --- GROUP 2: SYSTEM & NETWORK (Dependent on Time / Firmware) ---
# Model these using: Poisson(Time) or track as "Uptime %"
system_damages = [
    'Unresponsive Controller',     # Connectivity/Firmware (The big one)
    'Too many quick returns',      # User Behavior/System Flag
    'Unauthorized Trip',           # System/Theft Flag
    'Lock & unlock',               # Smart Lock Actuator/Comms
    'Lås',                         # Smart Lock Hardware
    'Console',                     # Dashboard Electronics
    'GPS',                         # Connectivity/Module
    'Battery',                     # Charging/BMS (Time + Cycles)
    'Vandalism'                    # External Factor (Scales with Time on Street)
]

In [183]:
# 2. Split the Data
damage_mechanical = damage_data[damage_data['damage_type_name'].isin(mechanical_damages)].copy()
damage_system = damage_data[damage_data['damage_type_name'].isin(system_damages)].copy()

# 3. Validation
print("--- SPLIT RESULTS ---")
print(f"Total Original Rows:   {len(damage_data):,}")
print(f"Mechanical Rows (Usage): {len(damage_mechanical):,}  (Use for Spare Parts Model)")
print(f"System Rows (Time):    {len(damage_system):,}  (Use for Reliability Model)")

# Check if any rows were lost (e.g. typos in the list vs actual data)
lost_rows = len(damage_data) - (len(damage_mechanical) + len(damage_system))
if lost_rows > 0:
    print(f"\n[WARNING] {lost_rows} rows were not categorized!")
    uncategorized = damage_data[~damage_data['damage_type_name'].isin(mechanical_damages + system_damages)]
    print("Uncategorized types found:", uncategorized['damage_type_name'].unique())
else:
    print("\n[SUCCESS] All rows successfully categorized.")

--- SPLIT RESULTS ---
Total Original Rows:   12,845
Mechanical Rows (Usage): 6,126  (Use for Spare Parts Model)
System Rows (Time):    6,719  (Use for Reliability Model)

[SUCCESS] All rows successfully categorized.


# Poisson regression model

## Aggregate the data

In [184]:


print(tabulate(trip_data.head(200), 
               headers='keys',       # <--- This is the missing fix
               tablefmt='psql', 
               showindex=False,
               floatfmt=".2f"))

+-----------+--------------+---------------------+----------------------------+----------------------------+--------------+--------------------------+----------------------------+-------------------------------+------------------------+--------------------------+-----------------------------+----------------------+------------------------------+
|   trip_id |   vehicle_id |   position_accuracy | trip_started_at            | trip_ended_at              | trip_state   |   trip_start_dock_number |   trip_start_dock_group_id | trip_start_dock_group_title   |   trip_end_dock_number |   trip_end_dock_group_id | trip_end_dock_group_title   |   segment_distance_m |   total_trip_distance_meters |
|-----------+--------------+---------------------+----------------------------+----------------------------+--------------+--------------------------+----------------------------+-------------------------------+------------------------+--------------------------+-----------------------------+-----------

In [185]:

# --- 2. AGGREGATE EXPOSURE (TRIPS) ---
print("\n" + "="*50)
print("--- STEP 2: AGGREGATING TRIPS (EXPOSURE) ---")

# A. Define "Legitimate Trips" at the individual trip level
# Criteria: Distance > 1km OR the bike moved to a different station
trip_data['is_legitimate_trip'] = (
    (trip_data['total_trip_distance_meters'] > 1000) | 
    (trip_data['trip_start_dock_group_id'] != trip_data['trip_end_dock_group_id'])
)

# B. Aggregate only legitimate trips by Month and Vehicle
trip_exposure = (
    trip_data[trip_data['is_legitimate_trip']]
    .set_index('trip_started_at')
    .groupby([pd.Grouper(freq='MS'), 'vehicle_id'])
    .agg({
        'total_trip_distance_meters': 'sum',
        'trip_id': 'count',
        'trip_start_dock_group_id': 'first',
        'trip_end_dock_group_id': 'last'
    })
    .reset_index()
    .rename(columns={
        'trip_started_at': 'month_start', 
        'total_trip_distance_meters': 'total_distance_m',
        'trip_id': 'trip_count'
    })
)

print(f"Total 'Vehicle-Months' with legitimate usage: {len(trip_exposure):,}")

# --- 3. AGGREGATE DAMAGES (SPLIT TARGETS) ---
print("\n--- STEP 3: AGGREGATING DAMAGES ---")

mech_counts = (
    damage_data[damage_data['damage_type_name'].isin(mechanical_damages)]
    .set_index('created_at')
    .groupby([pd.Grouper(freq='MS'), 'vehicle_id'])
    .size()
    .reset_index(name='damage_count')
    .rename(columns={'created_at': 'month_start'})
)

sys_counts = (
    damage_data[damage_data['damage_type_name'].isin(system_damages)]
    .set_index('created_at')
    .groupby([pd.Grouper(freq='MS'), 'vehicle_id'])
    .size()
    .reset_index(name='damage_count')
    .rename(columns={'created_at': 'month_start'})
)

# --- 4. PREPARE VEHICLE METADATA ---
print("\n--- STEP 4: PREPARING METADATA ---")
v_dmg = damage_data[['vehicle_id', 'vehicle_category', 'asset_model_id']]
v_mnt = maintenance_data[['vehicle_id', 'vehicle_category', 'asset_model_id']]
v_rpr = repair_data[['vehicle_id', 'vehicle_category', 'asset_model_id']]
vehicle_features = pd.concat([v_dmg, v_mnt, v_rpr]).drop_duplicates('vehicle_id')

# --- 5. BUILD MODEL A: MECHANICAL (DISTANCE BASED) ---
print("\n" + "="*50)
print("--- STEP 5: BUILDING MECHANICAL DATASET ---")

df_model_mech = pd.merge(trip_exposure, mech_counts, on=['month_start', 'vehicle_id'], how='left')
df_model_mech = df_model_mech.merge(vehicle_features, on='vehicle_id', how='left')

df_model_mech['damage_count'] = df_model_mech['damage_count'].fillna(0).astype(int)
df_model_mech['distance_km'] = df_model_mech['total_distance_m'] / 1000.0

# Filter out months with zero valid distance
df_model_mech = df_model_mech[df_model_mech['distance_km'] > 0].copy()

print(f"Final Mechanical Rows for Model: {len(df_model_mech):,}")

# DEBUG: Preview short but valid months
print("\n[DEBUG] Preview of months with < 1km but valid station moves:")
short_valid = df_model_mech[df_model_mech['distance_km'] < 1.0].head(10)
print(tabulate(short_valid[['month_start', 'vehicle_id', 'distance_km', 'trip_count', 'damage_count']], 
               headers='keys', tablefmt='psql', showindex=False))

# --- 6. BUILD MODEL B: SYSTEM (TIME BASED) ---
print("\n" + "="*50)
print("--- STEP 6: BUILDING SYSTEM DATASET ---")

# For the system model, we use the full trip_exposure (including all active months)
df_model_sys = pd.merge(trip_exposure, sys_counts, on=['month_start', 'vehicle_id'], how='left')
df_model_sys = df_model_sys.merge(vehicle_features, on='vehicle_id', how='left')

df_model_sys['damage_count'] = df_model_sys['damage_count'].fillna(0).astype(int)
df_model_sys['exposure_months'] = 1.0 

print(f"Final System Rows for Model: {len(df_model_sys):,}")
print("="*50)


--- STEP 2: AGGREGATING TRIPS (EXPOSURE) ---
Total 'Vehicle-Months' with legitimate usage: 4,632

--- STEP 3: AGGREGATING DAMAGES ---

--- STEP 4: PREPARING METADATA ---

--- STEP 5: BUILDING MECHANICAL DATASET ---
Final Mechanical Rows for Model: 4,609

[DEBUG] Preview of months with < 1km but valid station moves:
+---------------------+--------------+---------------+--------------+----------------+
| month_start         |   vehicle_id |   distance_km |   trip_count |   damage_count |
|---------------------+--------------+---------------+--------------+----------------|
| 2024-03-01 00:00:00 |          280 |    0.870148   |            2 |              0 |
| 2024-03-01 00:00:00 |         4067 |    0.668      |            1 |              0 |
| 2024-03-01 00:00:00 |         4278 |    0.996545   |            2 |              0 |
| 2024-03-01 00:00:00 |         4301 |    0.00497358 |            2 |              0 |
| 2024-03-01 00:00:00 |        62994 |    0.87138    |            2 |    

## Statistical modelling

Does the vehicle model or category significantly affect the breakdown rate per km?


### Mechanical model

Offset = distance in km

In [186]:
# Predicts: "Breakdowns per Km"
model_mech = smf.glm(
    formula="damage_count ~ 1", # Add categories here if you want: + C(vehicle_category)
    data=df_model_mech,
    offset=np.log(df_model_mech['distance_km']),
    family=sm.families.NegativeBinomial()
).fit()
print(model_mech.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           damage_count   No. Observations:                 4609
Model:                            GLM   Df Residuals:                     4608
Model Family:        NegativeBinomial   Df Model:                            0
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10412.
Date:                Mon, 16 Feb 2026   Deviance:                       13969.
Time:                        13:00:02   Pearson chi2:                 3.48e+05
No. Iterations:                    17   Pseudo R-squ. (CS):          2.054e-14
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -4.6370      0.022   -207.629      0.0

c:\Users\Minamsj\FOMOsim\.venv\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


### System model

- Driven by time. 
- The rate of ~1.1 errors per month suggests a stable electronic environment


In [187]:
# Predicts: "Errors per Active Month"
# Note: log(1) is 0, so the offset effectively does nothing, 
# which is correct for a "Per Month" rate.
model_sys = smf.glm(
    formula="damage_count ~ 1",
    data=df_model_sys,
    offset=np.log(df_model_sys['exposure_months']),
    family=sm.families.NegativeBinomial()
).fit()
print(model_sys.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           damage_count   No. Observations:                 4632
Model:                            GLM   Df Residuals:                     4631
Model Family:        NegativeBinomial   Df Model:                            0
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -6748.0
Date:                Mon, 16 Feb 2026   Deviance:                       5198.1
Time:                        13:00:02   Pearson chi2:                 7.01e+03
No. Iterations:                     5   Pseudo R-squ. (CS):         -4.441e-16
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.1004      0.020      4.949      0.0

c:\Users\Minamsj\FOMOsim\.venv\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
